In [0]:
from pyspark.sql.functions import col, current_timestamp, input_file_name, lit
from pyspark.sql.types import StructType, StructField, StringType

# 1. ARCHITECTURAL PILLAR: Late Binding Schema
# We define the schema as STRING to ensure even malformed backup data is captured
json_schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("amount", StringType(), True),
    StructField("country", StringType(), True),
    StructField("counterparty", StringType(), True),
    StructField("timestamp", StringType(), True)
])

# 2. EXTRACTION: Reading from the Backup Table
# Instead of readStream from Kafka, we load the static backup into df_raw
backup_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw_bkp"

df_raw = spark.read.table(backup_table) \
    .drop("id") \
    .withColumn("user_id", col("user_id").cast("string")) \
    .withColumn("amount", col("amount").cast("string")) \
    .withColumn("timestamp", col("timestamp").cast("string"))

# 3. ALGORITHMIC PILLAR: Routing & Validation
# Identify malformed records that caused previous issues
df_processed = df_raw.withColumn(
    "is_malformed",
    col("amount").cast("double").isNull() | col("user_id").cast("string").isNull()
)

# 4. KAPPA PILLAR: Writing to the "Source of Truth"
# We write to the primary Bronze table using mergeSchema to resolve the Double/String conflict
target_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw"

(df_processed
    .select("transaction_id", "user_id", "amount", "country", "counterparty", "timestamp", "ingestion_source")
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true") 
    .saveAsTable(target_table))

# 5. GOVERNANCE PILLAR: Populating the Quarantine Table
# Isolate the 'Poison Pills' for regulatory audit and manual review
df_quarantine = df_processed.filter(col("is_malformed") == True) \
    .select(
        "transaction_id",
        "user_id",
        "amount",
        "country",
        "counterparty",
        "timestamp"
    ) \
    .withColumn("quarantine_reason", lit("Schema Mismatch in Backup Recovery")) \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file_path", lit("backup_table"))

(df_quarantine.write
    .format("delta")
    .mode("append")
    .saveAsTable("`prism-sentinel-stream`.prism_bronze.transactions_quarantine"))

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit, when

# 1. PILLAR: LATE BINDING EXTRACTION
# We read from the backup table. Since we've already fixed the target 
# Bronze schema to STRING, we load this data to move it into the 'Source of Truth'.
backup_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw_bkp"
target_table = "`prism-sentinel-stream`.prism_bronze.transactions_raw"
quarantine_table = "`prism-sentinel-stream`.prism_bronze.transactions_quarantine"

df_backup = spark.read.table(backup_table) \
    .drop("id") \
    .withColumn("user_id", col("user_id").cast("string")) \
    .withColumn("amount", col("amount").cast("string")) \
    .withColumn("timestamp", col("timestamp").cast("string"))

# 2. PILLAR: ALGORITHMIC VALIDATION (The "Evasion Hunter" Preparation)
# We identify records where 'user_id' or 'amount' are physically BYTE_ARRAY/String 
# but contain non-numeric data. This prevents the downstream Monte Carlo 
# simulations from receiving "Null" values that skew risk results.
df_validated = df_backup.withColumn(
    "is_malformed",
    col("amount").cast("double").isNull() | col("user_id").cast("double").isNull()
).withColumn(
    "quarantine_reason",
    when(col("amount").cast("double").isNull(), "Non-numeric Amount")
    .when(col("user_id").cast("double").isNull(), "Non-numeric UserID")
    .otherwise(lit(None))
)

# 3. PILLAR: KAPPA ARCHITECTURE RE-ENTRY
# Append all records to the main Bronze table. We use mergeSchema=True 
# to ensure the transition from the old Double types to new String types is seamless.
(df_validated.drop("quarantine_reason")
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true") 
    .saveAsTable(target_table))

# 4. PILLAR: REGULATORY COMPLIANCE (Quarantine Population)
# Isolate the malformed data for the Data Governance 'Evidence Locker'.
df_quarantine = df_validated.filter(col("is_malformed") == True) \
    .select(
        "transaction_id",
        "user_id",
        "amount",
        "country",
        "counterparty",
        "timestamp",
        "quarantine_reason"
    ) \
    .withColumn("ingestion_time", current_timestamp()) \
    .withColumn("source_file_path", lit("backup_table"))

(df_quarantine.write
    .format("delta")
    .mode("append")
    .saveAsTable(quarantine_table))

print(f"Success: Data recovered to {target_table}. Malformed records isolated in {quarantine_table}.")

In [0]:
%sql
SELECT *
FROM `prism-sentinel-stream`.prism_bronze.transactions_quarantine

In [0]:
%sql
UPDATE `prism-sentinel-stream`.prism_bronze.transactions_raw
SET 
  is_malformed = (try_cast(amount AS DOUBLE) IS NULL OR try_cast(user_id AS DOUBLE) IS NULL)
WHERE is_malformed IS NULL; -- Only target the 40M legacy records

In [0]:
%sql
INSERT INTO `prism-sentinel-stream`.prism_bronze.transactions_quarantine
(transaction_id, user_id, amount, country, counterparty, timestamp,ingestion_time,source_file_path,is_resolved)
SELECT 
  transaction_id, user_id, amount, country, counterparty, timestamp,
  --quarantine_reason,
  current_timestamp() as ingestion_time,
  'backfill_reprocess' as source_file_path,
  false as is_resolved
FROM `prism-sentinel-stream`.prism_bronze.transactions_raw tr
WHERE is_malformed = true 
AND NOT EXISTS (SELECT transaction_id FROM `prism-sentinel-stream`.prism_bronze.transactions_quarantine tq
WHERE tq.transaction_id = tr.transaction_id);

In [0]:
%sql
SELECT 
    is_malformed,
    count(*) as record_count,
    round(count(*) * 100.0 / sum(count(*)) over(), 6) as distribution_pct
FROM `prism-sentinel-stream`.prism_bronze.transactions_raw
GROUP BY is_malformed;